# Smart MCQ Solver (Gemma 4 Fine-tuned)

This notebook contains the codes for fine-tunning Gemma 4.

## Build By:

- **Name:** Varun Agnihotri
- **Roll Number:** `24f2004142`

## Setup

In [ ]:
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!uv pip install \
    unsloth==2026.6.7 \
    unsloth-zoo==2026.6.5 \
    triton==3.6.0 \
    transformers==5.5.0 \
    peft==0.18.1 \
    bitsandbytes==0.49.2 \
    trl==0.24.0 \
    datasets==4.3.0 \
    xformers==0.0.35 \
    torchao==0.17.0

## Imports

In [ ]:
from unsloth import FastLanguageModel

In [ ]:
import math
import pickle
import re
import warnings
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
import wandb
from datasets import Dataset
from kaggle_secrets import UserSecretsClient  # type: ignore
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from trl import SFTConfig, SFTTrainer

In [ ]:
warnings.filterwarnings("ignore")

## Configurations

In [ ]:
user_secrets = UserSecretsClient()

In [ ]:
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

In [ ]:
# Dataset Path
DATA_DIR = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"

# Output paths
TRAIN_OUT_DIR = Path("/kaggle/working/gemma4-mcq-lora/")
TRAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL_OUT_DIR = Path("/kaggle/working/gemma4-base/")
# BASE_MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

FINETUNE_MODEL_OUT_DIR = Path("/kaggle/working/gemma4-lora/")
FINETUNE_MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Model
MODEL_PATH = Path("/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1")
MAX_SEQ_LEN = 512
LOAD_IN_4BIT = True

# Lora
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0

# Training
VAL_FRAC = 0.2
EPOCHS = 10
BATCH_SIZE = 1
GRAD_ACCUM = 16
LR = 2e-4
WARMUP_RATIO = 0.05
LR_SCHEDULER = "cosine"

# Self RAG
TOP_K = 3

# Seed
SEED = 42

# Valid Choices
CHOICES = ["A", "B", "C", "D", "E"]

## Wandb Config

In [ ]:
wandb.init(
    entity=user_secrets.get_secret("WANDB_ENTITY"),
    project=user_secrets.get_secret("WANDB_PROJECT"),
    notes="Gemma 4 E4B IT Finetune",
    config={
        # Paths
        "model_path": str(MODEL_PATH),
        "train_csv": str(TRAIN_CSV),
        "test_csv": str(TEST_CSV),
        # Model
        "max_seq_len": MAX_SEQ_LEN,
        "load_in_4bit": LOAD_IN_4BIT,
        # LoRA
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        # Training
        "val_frac": VAL_FRAC,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "effective_batch": BATCH_SIZE * GRAD_ACCUM,
        "lr": LR,
        "warmup_ratio": WARMUP_RATIO,
        "lr_scheduler": LR_SCHEDULER,
        # Self-RAG
        "top_k_rag": TOP_K,
        # Misc
        "seed": SEED,
        "choices": CHOICES,
    },
)

## Helper Functions

In [ ]:
def map_at_3(preds: List[List[str]], labels: List[str]) -> float:
    score = 0.0
    for pred, label in zip(preds, labels):
        for k, p in enumerate(pred[:3], 1):
            if p == label:
                score += 1.0 / k
                break

    return score / max(1, len(labels))

In [ ]:
def set_seed(s: int):
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

In [ ]:
def build_user_message(row: Dict, context: str = "") -> str:
    """
    Builds the user-turn message.

    With context (self RAG):
      Context:
      Q: <similar question>
      Answer: A) <text>

      Q: ...
      Answer: ...

      Question: <current question>
      A) ...   B) ...   C) ...   D) ...   E) ...
      The correct answer is:

    Without context (zero-shot):
      Question: <current question>
      ...
    """
    context_block = f"Context:\n{context.strip()}\n\n" if context.strip() else ""
    return (
        f"{context_block}"
        f"Question: {row['prompt']}\n\n"
        f"A) {row['A']}\n"
        f"B) {row['B']}\n"
        f"C) {row['C']}\n"
        f"D) {row['D']}\n"
        f"E) {row['E']}\n\n"
        f"The correct answer is:"
    )

In [ ]:
def format_for_sft(row: Dict, tokenizer, context: str = "") -> Dict:
    context = str(row.get("_context", ""))
    messages = [
        {"role": "user", "content": build_user_message(row, context)},
        {"role": "assistant", "content": str(row["answer"]).strip().upper()},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

In [ ]:
def clean(text: str) -> str:
    text = str(text).lower()  # type: ignore
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
def extract_core_question(prompt: str) -> str:
    """Extract core question by removing template prefixes and suffixes."""
    if ":" in prompt:
        q = prompt.split(":", 1)[1].strip()
    else:
        q = prompt.strip()

    if "?" in q:
        q = q.rsplit("?", 1)[0].strip() + "?"
    return clean(q)

In [ ]:
@torch.inference_mode()
def score_and_rank(
    model,
    tokenizer,
    row: Dict,
    choice_token_ids: Dict[str, int],
    choices: List[str],
) -> List[str]:
    """
    Single forward pass - log-prob ranking of all 5 choices.
    """
    context = str(row.get("_context", ""))
    messages = [{"role": "user", "content": build_user_message(row, context)}]
    prefix_str = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # appends <start_of_turn>model\n
    )
    # print(prefix_str)
    input_ids = tokenizer(
        text=prefix_str,
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids.to(model.device)

    # Forward pass
    last_logits = model(input_ids).logits[0, -1, :]  # [vocab_size]
    log_probs = torch.log_softmax(last_logits, dim=-1)

    scores = {ch: log_probs[tid].item() for ch, tid in choice_token_ids.items()}
    ranked = sorted(scores, key=lambda k: scores[k], reverse=True)
    return ranked[:3]

In [ ]:
def get_choice_token_ids(tokenizer, choices: List[str]) -> Dict[str, int]:
    """
    Resolves each answer label to its single vocabulary token ID.
    """
    choice_ids = {}
    for ch in choices:
        plain = tokenizer.tokenizer.encode(ch, add_special_tokens=False)
        spaced = tokenizer.tokenizer.encode(" " + ch, add_special_tokens=False)
        if len(plain) == 1:
            choice_ids[ch] = plain[0]
        elif len(spaced) == 1:
            choice_ids[ch] = spaced[0]
        else:
            choice_ids[ch] = plain[0]
            print(f"Warning: '{ch}' is multi-token ({plain}). Using first token.")

    decoded = {k: (v, tokenizer.tokenizer.decode([v])) for k, v in choice_ids.items()}
    print(f"Choice token IDs: {decoded}")
    return choice_ids

In [ ]:
def evaluate(
    model,
    tokenizer,
    val_df: pd.DataFrame | pd.Series,
    label: str = "Evaluation",
) -> Dict[str, float]:
    """
    Runs inference on the val set (using pre-computed contexts
    from val_df["_context"]) and reports MAP@3 + accuracy breakdown.
    """
    FastLanguageModel.for_inference(model)
    model.eval()

    choice_ids = get_choice_token_ids(tokenizer, CHOICES)
    predictions, labels = [], []

    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc=label):
        top3 = score_and_rank(model, tokenizer, row.to_dict(), choice_ids, CHOICES)
        predictions.append(top3)
        labels.append(str(row["answer"]).strip().upper())

    map3 = map_at_3(predictions, labels)

    true_idx = [CHOICES.index(lbl) for lbl in labels]
    pred_idx = [CHOICES.index(p[0]) for p in predictions]

    metrics = {
        "val_map3": map3,
        "val_accuracy": float(np.mean(np.array(pred_idx) == np.array(true_idx))),
        "val_macro_f1": float(f1_score(true_idx, pred_idx, average="macro")),
    }

    n = len(labels)
    acc = {k: sum(a in p[:k] for a, p in zip(labels, predictions)) for k in [1, 2, 3]}

    print(f"\n{'-' * 45}")
    print(f"  {label}")
    print(f"{'-' * 45}")
    print(f"  MAP@3       :  {metrics['val_map3']:.4f}")
    print(f"  Macro F1    :  {metrics['val_macro_f1']:.4f}")
    print(f"  Accuracy@1  :  {acc[1] / n:.4f}  ({acc[1]}/{n})")
    print(f"  Accuracy@2  :  {acc[2] / n:.4f}  ({acc[2]}/{n})")
    print(f"  Accuracy@3  :  {acc[3] / n:.4f}  ({acc[3]}/{n})")
    print(f"{'-' * 45}\n")
    return metrics

## Retriever

In [ ]:
class TFIDFRetriever:
    """
    TF-IDF retriever over the training corpus that provides few-shot
    context for each question.
    """

    def __init__(self, df: pd.DataFrame | pd.Series, top_k: int = 3) -> None:
        self.top_k = top_k
        self.corpus = df.reset_index(drop=True).copy()
        self.corpus["_core"] = self.corpus["prompt"].apply(extract_core_question)

        self.vectorizer = TfidfVectorizer(
            max_features=50_000,
            ngram_range=(1, 2),
            sublinear_tf=True,
            strip_accents="unicode",
            token_pattern=r"(?u)\b\w\w+\b",
        )
        self.corpus_matrix = self.vectorizer.fit_transform(self.corpus["_core"])
        print(
            f"TF-IDF index: {self.corpus_matrix.shape[0]} documents, "
            f"{self.corpus_matrix.shape[1]} features"
        )

    def retrieve(
        self,
        query: str,
        exclude_idx: Optional[int] = None,
    ) -> str:
        """
        Returns a formatted context string for one query prompt.

        Returned format (3 retrieved pairs, separated by blank lines):
          Q: <similar question>
          Answer: <label>) <full answer text>

          Q: ...
          Answer: ...
        """
        core_q = extract_core_question(query)
        qv = self.vectorizer.transform([core_q])
        sims = cosine_similarity(qv, self.corpus_matrix).flatten()

        is_sibling = self.corpus["_core"].to_numpy() == core_q
        sims = sims + is_sibling * 10.0  # exact core have higher similarity

        order = sims.argsort()[::-1]
        ctx_parts = []
        for idx in order:
            if idx == exclude_idx:
                continue

            row = self.corpus.iloc[idx]
            label = str(row["answer"]).strip().upper()
            answer_text = str(row[label]).strip()
            ctx_parts.append(f"Q: {row['prompt']}\nAnswer: {label}) {answer_text}")
            if len(ctx_parts) >= self.top_k:
                break

        return "\n\n".join(ctx_parts)

    def precompute_contexts(
        self,
        df: pd.DataFrame | pd.Series,
        desc: str,
        is_corpus: bool = False,
    ) -> List[str]:
        contexts = []
        rows = tqdm(df.iterrows(), total=len(df), desc=desc)
        for pos, (_, row) in enumerate(rows):
            contexts.append(
                self.retrieve(
                    row["prompt"],
                    exclude_idx=pos if is_corpus else None,
                )
            )
        return contexts

    def save(self, path: str | Path):
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path: str | Path):
        with open(path, "rb") as f:
            return pickle.load(f)

## Pre-processing

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["core_question"] = df["prompt"].apply(extract_core_question)

In [ ]:
rng = np.random.default_rng(SEED)
order = rng.permutation(len(df))

In [ ]:
n_val = max(1, int(len(df) * VAL_FRAC))
val_idx = set(order[:n_val].tolist())

In [ ]:
val_mask = df.index.isin(val_idx)
df_val = df[val_mask].reset_index(drop=True)
df_train = df[~val_mask].reset_index(drop=True)

print(f"Data Split: train={len(df_train)} samples, validation={len(df_val)} samples")

In [ ]:
print("Fitting TF-IDF retriever on train split...")
retriever = TFIDFRetriever(df_train, top_k=TOP_K)

In [ ]:
print("Pre-computing RAG contexts for data")

df_train["_context"] = retriever.precompute_contexts(
    df_train,
    "Retrieving contexts for train split",
    is_corpus=True,
)
df_val["_context"] = retriever.precompute_contexts(
    df_val, "Retrieving contexts for val split"
)

In [ ]:
df_train.sample(5)

In [ ]:
df_val.sample(5)

## Model Loading

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(MODEL_PATH),
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    device_map={"": 0},
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
model.config.use_cache = False  # type: ignore

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"

In [ ]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}  ({100 * trainable / total:.2f}%)\n")

## Zero-shot Baseline

In [ ]:
df_val_without_context = df_val.copy()
df_val_without_context["_context"] = ""

evaluate(model, tokenizer, df_val_without_context, label="Zero-shot + No context")

In [ ]:
evaluate(model, tokenizer, df_val, label="Zero-shot + context")

In [ ]:
# Save the base model to BASE_MODEL_OUT_DIR
# shutil.copytree(MODEL_PATH, BASE_MODEL_OUT_DIR)

## Fine-tuning

In [ ]:
FastLanguageModel.for_training(model)

In [ ]:
formatted_rows = [
    format_for_sft(row.to_dict(), tokenizer) for _, row in df_train.iterrows()
]
train_dataset = Dataset.from_list(formatted_rows)

In [ ]:
total_steps = math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

In [ ]:
training_args = SFTConfig(
    output_dir=str(TRAIN_OUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=warmup_steps,
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    save_strategy="epoch",
    save_total_limit=1,
    seed=SEED,
    report_to="wandb",
    logging_steps=20,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,  # type: ignore
)

In [ ]:
trainer = SFTTrainer(
    model=model,  # type: ignore
    tokenizer=tokenizer,  # type: ignore
    train_dataset=train_dataset,
    args=training_args,
)

In [ ]:
print(
    f"Training:  {len(train_dataset)} examples  |  "
    f"{EPOCHS} epochs  |  "
    f"effective batch = {BATCH_SIZE * GRAD_ACCUM}\n"
)
trainer.train()
print("Training complete.\n")

In [ ]:
metrics = evaluate(model, tokenizer, df_val, label="Fine-tuned + context")
metrics

In [ ]:
wandb.log(metrics)
if wandb.run is not None:
    wandb.run.summary.update(metrics)
    wandb.run.summary["best_val_map3"] = metrics["val_map3"]
    wandb.run.summary["best_val_accuracy"] = metrics["val_accuracy"]
    wandb.run.summary["best_val_macro_f1"] = metrics["val_macro_f1"]

wandb.finish()

## Save Model

In [ ]:
# merged_model = model.merge_and_unload()  # type: ignore
# merged_model.save_pretrained(str(FINETUNE_MODEL_OUT_DIR), safe_serialization=True)
model.save_pretrained(str(FINETUNE_MODEL_OUT_DIR))  # type: ignore

In [ ]:
tokenizer.save_pretrained(FINETUNE_MODEL_OUT_DIR)

In [ ]:
print("Fitting TF-IDF retriever on full dataset...")
retriever = TFIDFRetriever(df, top_k=TOP_K)
retriever.save(FINETUNE_MODEL_OUT_DIR / "retriever.pkl")